# Feature Engineering


In [12]:
import pandas as pd

In [13]:
df = pd.read_csv("../data/interim/merged_external_dataset.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.sort_values("timestamp").reset_index(drop=True)

df.head()

,timestamp,price,actual_load,actual_residual_load,forecasted_load,forecasted_residual_load,actual_biomass,actual_hydropower,actual_wind_offshore,actual_wind_onshore,actual_solar,actual_lignite,actual_hard_coal,actual_fossil_gas,actual_wind_total,forecasted_solar_wind_total,forecasted_wind_offshore,forecasted_wind_onshore,forecasted_solar,forecasted_wind_total
0,2023-01-01 00:00:00,-5.17,38346.00,6337.75,41792.50,2798.75,4014.25,1275.25,3059.25,28947.25,1.75,3859.25,2067.50,1593.75,32006.50,38993.75,3478.25,35515.50,0.0,38993.75
1,2023-01-01 01:00:00,-1.07,37777.25,4601.75,39621.00,886.25,3993.25,1226.50,3586.00,29587.50,2.00,3866.50,2052.00,1437.00,33173.50,38734.75,3390.25,35344.50,0.0,38734.75
2,2023-01-01 02:00:00,-1.47,36939.75,3581.00,38240.75,-293.50,3967.25,1222.50,3842.25,29514.75,1.75,3860.25,2034.25,1435.00,33357.00,38534.25,3395.50,35138.75,0.0,38534.25
3,2023-01-01 03:00:00,-5.08,35932.50,4973.75,37205.50,-645.75,3973.00,1223.25,3463.25,27493.50,2.00,3864.75,2037.00,1432.50,30956.75,37851.25,3410.25,34441.00,0.0,37851.25
4,2023-01-01 04:00:00,-4.49,35486.25,5082.75,37326.75,-2.50,3996.50,1244.00,3462.25,26939.00,2.25,3841.00,2040.25,1430.75,30401.25,37329.25,3431.25,33898.00,0.0,37329.25


In [14]:
df = df.set_index("timestamp")

In [15]:
#Time features
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month

import numpy as np

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

#Lag features
df["lag_24"] = df["price"].shift(24)
df["lag_48"] = df["price"].shift(48)
df["lag_168"] = df["price"].shift(168)

#Rolling features
df["rolling_mean_24"] = df["price"].shift(24).rolling(24).mean()
df["rolling_std_24"] = df["price"].shift(24).rolling(24).std()


In [16]:
# Total renewable generation
df["actual_renewable_total"] = (
    df["actual_wind_total"] +
    df["actual_solar"] +
    df["actual_hydropower"] +
    df["actual_biomass"]
)

# Total fossil generation
df["actual_fossil_total"] = (
    df["actual_lignite"] +
    df["actual_hard_coal"] +
    df["actual_fossil_gas"]
)

In [17]:
external_cols = [
    "actual_load",
    "actual_residual_load",
    "actual_wind_total",
    "actual_solar",
    "actual_renewable_total",
    "actual_fossil_total"
]

for col in external_cols:
    df[f"{col}_lag_24"] = df[col].shift(24)
    df[f"{col}_lag_48"] = df[col].shift(48)
    df[f"{col}_lag_168"] = df[col].shift(168)

In [18]:
df_features = df.dropna().copy()

df_features.shape

(27093, 49)

In [19]:
df_features.to_csv("../data/processed/features_dataset.csv")

print("Feature dataset saved")
print("Shape:", df_features.shape)

Feature dataset saved
Shape: (27093, 49)
